# Agent Memory Toolkit – Async Demo

This notebook demonstrates the **async** API of the Agent Memory Toolkit via `AsyncCosmosMemoryClient`:

1. [**Setup**](#1-setup) – Install dependencies and load environment variables
2. [**Local memory operations**](#2-local-memory-operations) – `add_local`, `get_local`, `update_local`, `delete_local`
3. [**Cosmos DB operations**](#3-cosmos-db-operations) – `add_cosmos`, `get_memories`
4. [**Summarization**](#4-summarization) – `generate_thread_summary()` with `AsyncCosmosMemoryClient`
5. [**Fact Extraction**](#5-fact-extraction) – `extract_facts()` with `AsyncCosmosMemoryClient`
6. [**User Summary**](#6-user-summary) – `generate_user_summary()` and `get_user_summary()` with `AsyncCosmosMemoryClient`
7. [**Vector search**](#7-vector-search-with-`search_cosmos`) – `search_cosmos()` with `AsyncCosmosMemoryClient`
8. [**Automatic processing (Change Feed)**](#8-automatic-processing-change-feed) – Write turns and let the change feed trigger process them automatically

> **Local hosting:** Sections 4–7 require the Azure Durable Functions host running locally via `func start` (see [local_testing.md](../Docs/local_testing.md) for setup). Section 8 additionally requires the change feed settings configured in `local.settings.json`. To run against a deployed Function App instead, update `ADF_ENDPOINT` in your `.env` file.

> For the **sync** API (`CosmosMemoryClient`), see [Demo.ipynb](Demo.ipynb).

## 1. Setup

Install/import dependencies and load environment variables from `.env`.

In [1]:
import os, json, sys
from pathlib import Path
sys.path.insert(0, os.path.abspath(".."))

from dotenv import load_dotenv
from agent_memory_toolkit.aio import AsyncCosmosMemoryClient

# Load environment variables from .env in the repo root
load_dotenv(os.path.join("..", ".env"))

print("COSMOS_DB_ENDPOINT:", os.getenv("COSMOS_DB_ENDPOINT"))
print("COSMOS_DB_DATABASE:", os.getenv("COSMOS_DB_DATABASE"))
print("COSMOS_DB_CONTAINER:", os.getenv("COSMOS_DB_CONTAINER"))

COSMOS_DB_ENDPOINT: https://akataria-agent-memory-testing.documents.azure.com:443/
COSMOS_DB_DATABASE: ai_memory
COSMOS_DB_CONTAINER: memories


In [2]:
from azure.identity.aio import DefaultAzureCredential as AsyncDefaultAzureCredential

# Create an AsyncCosmosMemoryClient instance
memory = AsyncCosmosMemoryClient(
    cosmos_endpoint=os.getenv("COSMOS_DB_ENDPOINT"),
    cosmos_database=os.getenv("COSMOS_DB_DATABASE"),
    cosmos_container=os.getenv("COSMOS_DB_CONTAINER"),
    ai_foundry_endpoint=os.getenv("AI_FOUNDRY_ENDPOINT"),
    embedding_model=os.getenv("EMBEDDING_MODEL", "text-embedding-3-large"),
    adf_endpoint=os.getenv("ADF_ENDPOINT", "http://localhost:7071/api"),
    adf_key=os.getenv("ADF_KEY", ""),
    use_default_credential=True,
    cosmos_credential=AsyncDefaultAzureCredential(),
)

print("AsyncCosmosMemoryClient instance created")
print("Local memory store:", memory.local_memory)

AsyncCosmosMemoryClient instance created
Local memory store: []


## 2. Local Memory Operations

`AsyncCosmosMemoryClient` mirrors the sync API. Local operations (`add_local`, `get_local`, `update_local`, `delete_local`) are synchronous under the hood (in-memory list), but the class is designed for use in async code paths.

> **Note:** In Jupyter you can `await` directly in cells without wrapping in `asyncio.run()`.

### 2a. Add memories with `add_local`

In [3]:
import uuid
THREAD_ID = str(uuid.uuid4())
print(f"Thread ID: {THREAD_ID}\n")

Thread ID: c55ab0cb-326a-478a-aaa2-af1469480f58



In [4]:
# Add sample conversation: weather in Seattle → booking a trip (6 turns)
memory.add_local(
    user_id="user-001", role="user", thread_id=THREAD_ID,
    content="What's the weather like in Seattle this weekend?",
)
memory.add_local(
    user_id="user-001", role="agent", thread_id=THREAD_ID,
    content="This weekend Seattle will be around 55°F with partly cloudy skies on Saturday and light rain expected Sunday.",
)
memory.add_local(
    user_id="user-001", role="user", thread_id=THREAD_ID,
    content="That sounds nice enough. Can you help me book a trip to Seattle for this weekend?",
)
memory.add_local(
    user_id="user-001", role="agent", thread_id=THREAD_ID,
    content="Sure! I found round-trip flights departing Friday evening and returning Sunday night. There are also several hotels in downtown Seattle with availability. Would you like me to look at specific airlines or neighborhoods?",
)
memory.add_local(
    user_id="user-001", role="user", thread_id=THREAD_ID,
    content="Something near Pike Place Market would be great. And keep the flights under $300 round trip if possible.",
)
memory.add_local(
    user_id="user-001", role="agent", thread_id=THREAD_ID,
    content="I found a round-trip on Alaska Airlines for $275 and two hotel options within a 5-minute walk of Pike Place Market: the Inn at the Market ($189/night) and a Hilton Garden Inn ($145/night). Want me to reserve one of these?",
)

print(f"Added {len(memory.local_memory)} memories")
print(json.dumps(memory.get_local(), indent=2))

Added 6 memories
[
  {
    "id": "59ba9c05-a475-4c7c-b13b-1ceedf4c9332",
    "user_id": "user-001",
    "thread_id": "c55ab0cb-326a-478a-aaa2-af1469480f58",
    "role": "user",
    "type": "turn",
    "content": "What's the weather like in Seattle this weekend?",
    "metadata": {},
    "created_at": "2026-04-07T22:06:47.685709+00:00"
  },
  {
    "id": "fed9f5ee-c1b4-4378-a217-8bab2332c087",
    "user_id": "user-001",
    "thread_id": "c55ab0cb-326a-478a-aaa2-af1469480f58",
    "role": "agent",
    "type": "turn",
    "content": "This weekend Seattle will be around 55\u00b0F with partly cloudy skies on Saturday and light rain expected Sunday.",
    "metadata": {},
    "created_at": "2026-04-07T22:06:47.685750+00:00"
  },
  {
    "id": "24ca3052-a0bb-44f3-82c1-513b89d7d28a",
    "user_id": "user-001",
    "thread_id": "c55ab0cb-326a-478a-aaa2-af1469480f58",
    "role": "user",
    "type": "turn",
    "content": "That sounds nice enough. Can you help me book a trip to Seattle for this w

### 2b. Query memories with `get_local`

In [5]:
# Get all memories
all_memories = memory.get_local()
print(f"Total memories: {len(all_memories)}\n")

# Filter by user_id
user1_memories = memory.get_local(user_id="user-001")
print(f"Memories for user-001: {len(user1_memories)}")

# Filter by role
tool_memories = memory.get_local(role="agent")
print(f"Tool memories: {len(tool_memories)}")
for m in tool_memories:
    print(f"  [{m['id'][:8]}...] {m['content'][:60]}")

# Filter by type
facts = memory.get_local(memory_type="fact")
print(f"\nFact memories: {len(facts)}")
for m in facts:
    print(f"  [{m['id'][:8]}...] {m['content']}")

# Combine filters: user-001 + agent role
user1_agent = memory.get_local(user_id="user-001", role="agent")
print(f"\nAgent memories for user-001: {len(user1_agent)}")

Total memories: 6

Memories for user-001: 6
Tool memories: 3
  [fed9f5ee...] This weekend Seattle will be around 55°F with partly cloudy 
  [98c43caa...] Sure! I found round-trip flights departing Friday evening an
  [b99ae6ee...] I found a round-trip on Alaska Airlines for $275 and two hot

Fact memories: 0

Agent memories for user-001: 3


### 2c. Update & Delete with `update_local` / `delete_local`

In [6]:
# Update the user's budget constraint to be more specific
target_id = memory.local_memory[4]["id"]  # "Something near Pike Place Market..."
print(f"Before update:\n{json.dumps(memory.get_local(memory_id=target_id)[0], indent=2)}\n")

memory.update_local(
    memory_id=target_id,
    content="Something near Pike Place Market would be great. Keep flights under $300 round trip and hotels under $200/night.",
    metadata={"edited": True, "reason": "user clarified hotel budget"},
)
print(f"After update:\n{json.dumps(memory.get_local(memory_id=target_id)[0], indent=2)}")

# Delete the third memory (index 2 — the user's booking request)
del_target_id = memory.local_memory[2]["id"]
print(f"\nDeleting memory {del_target_id[:8]}...")
memory.delete_local(del_target_id)

# Verify it's gone
print(f"\nRemaining memories: {len(memory.get_local())}")
for m in memory.get_local():
    print(f" [{m['thread_id'][:8]}...]  [{m['id'][:8]}...] role={m['role']:<6} type={m['type']:<8} {m['content'][:50]}")

Before update:
{
  "id": "6e10d230-1b1d-4a8b-b3e0-47b8e03bd13f",
  "user_id": "user-001",
  "thread_id": "c55ab0cb-326a-478a-aaa2-af1469480f58",
  "role": "user",
  "type": "turn",
  "content": "Something near Pike Place Market would be great. And keep the flights under $300 round trip if possible.",
  "metadata": {},
  "created_at": "2026-04-07T22:06:47.685820+00:00"
}

After update:
{
  "id": "6e10d230-1b1d-4a8b-b3e0-47b8e03bd13f",
  "user_id": "user-001",
  "thread_id": "c55ab0cb-326a-478a-aaa2-af1469480f58",
  "role": "user",
  "type": "turn",
  "content": "Something near Pike Place Market would be great. Keep flights under $300 round trip and hotels under $200/night.",
  "metadata": {
    "edited": true,
    "reason": "user clarified hotel budget"
  },
  "created_at": "2026-04-07T22:06:47.685820+00:00",
  "updated_at": "2026-04-07T22:06:53.579267+00:00"
}

Deleting memory 24ca3052...

Remaining memories: 5
 [c55ab0cb...]  [59ba9c05...] role=user   type=turn     What's the weather 

## 3. Cosmos DB Operations

### 3a. Connect and create the memory store

The async client auto-connects on the first Cosmos DB operation. Call `create_memory_store()` to create the database and container if they do not already exist, including the hierarchical partition key, vector index, and full-text index.

> **Note:** `create_memory_store()` is safe to run more than once.

In [7]:
# The async client auto-creates the database and container on the first Cosmos operation.
# You can also call create_memory_store() explicitly if needed.
await memory.create_memory_store()
print(f"Connected: {memory._container_client is not None}")

Connected: True


### 3b. Add memories to Cosmos DB with `add_cosmos`

In [8]:
await memory.push_to_cosmos()

# Push a new thread directly to Cosmos DB without adding to local memory first
new_thread_id = str(uuid.uuid4())
print(f"New Thread ID: {new_thread_id}\n")

# Add memories directly to Cosmos DB using add_cosmos
await memory.add_cosmos(
    user_id="user-002", role="user", thread_id=new_thread_id,
    content="Can you recommend some good restaurants in New York City?",
)
await memory.add_cosmos(
    user_id="user-002", role="tool", thread_id=new_thread_id,
    content='{"query": "top restaurants NYC", "results": ["Carbone", "Nobu", "Katz\'s Deli", "Le Bernardin"]}',
    metadata={"tool_name": "restaurant_search", "tool_call_id": "call_abc123"},
)
await memory.add_cosmos(
    user_id="user-002", role="agent", thread_id=new_thread_id,
    content="Absolutely! NYC has incredible dining options. For Italian, try Carbone in Greenwich Village. For sushi, Nobu in Tribeca is world-class. For a classic NYC experience, Katz's Delicatessen on the Lower East Side is a must.",
)
await memory.add_cosmos(
    user_id="user-002", role="user", thread_id=new_thread_id,
    content="I love Italian food. Are there any options that are budget-friendly?",
)
await memory.add_cosmos(
    user_id="user-002", role="agent", thread_id=new_thread_id,
    content="For budget-friendly Italian in NYC, check out L'industrie Pizzeria in Williamsburg or Artichoke Basille's Pizza. Both are highly rated and won't break the bank.",
)

# Verify the memories were added directly to Cosmos DB (not in local memory)
print(f"Local memory count (should be unchanged): {len(memory.local_memory)}\n")

cosmos_results = await memory.get_memories(user_id="user-002", thread_id=new_thread_id)
print(f"Memories in Cosmos DB for new thread: {len(cosmos_results)}")
for r in cosmos_results:
    print(f"  [{r['thread_id'][:8]}...] [{r['id'][:8]}...] role={r['role']:<6} {r['content'][:60]}")

New Thread ID: 5f86030e-bc73-4fba-bdb5-5a9c8404f9d4

Local memory count (should be unchanged): 5

Memories in Cosmos DB for new thread: 5
  [5f86030e...] [31425499...] role=user   Can you recommend some good restaurants in New York City?
  [5f86030e...] [807a2774...] role=tool   {"query": "top restaurants NYC", "results": ["Carbone", "Nob
  [5f86030e...] [ab8f1310...] role=agent  Absolutely! NYC has incredible dining options. For Italian, 
  [5f86030e...] [cba2601c...] role=user   I love Italian food. Are there any options that are budget-f
  [5f86030e...] [f209986d...] role=agent  For budget-friendly Italian in NYC, check out L'industrie Pi


### 3c. Retrieve memories from Cosmos DB with `get_memories`

Supports the same filters as `get_local`: `memory_id`, `user_id`, `role`, `memory_type`.

In [9]:
# Get all memories for user-001
results = await memory.get_memories(user_id="user-001")
print(f"Memories for user-001: {len(results)}\n")
for r in results:
    print(f"  [{r['thread_id'][:8]}...] [{r['id'][:8]}...] role={r['role']:<6} type={r['type']:<8} {r['content'][:50]}")

# Get only agent memories
agent_results = await memory.get_memories(role="agent")
print(f"\nAgent memories: {len(agent_results)}")
for r in agent_results:
    print(f"  [{r['thread_id'][:8]}...] [{r['id'][:8]}...] role={r['role']:<6} type={r['type']:<8} {r['content'][:50]}")

Memories for user-001: 5

  [c55ab0cb...] [98c43caa...] role=agent  type=turn     Sure! I found round-trip flights departing Friday 
  [c55ab0cb...] [6e10d230...] role=user   type=turn     Something near Pike Place Market would be great. K
  [c55ab0cb...] [fed9f5ee...] role=agent  type=turn     This weekend Seattle will be around 55°F with part
  [c55ab0cb...] [b99ae6ee...] role=agent  type=turn     I found a round-trip on Alaska Airlines for $275 a
  [c55ab0cb...] [59ba9c05...] role=user   type=turn     What's the weather like in Seattle this weekend?

Agent memories: 5
  [c55ab0cb...] [98c43caa...] role=agent  type=turn     Sure! I found round-trip flights departing Friday 
  [c55ab0cb...] [fed9f5ee...] role=agent  type=turn     This weekend Seattle will be around 55°F with part
  [c55ab0cb...] [b99ae6ee...] role=agent  type=turn     I found a round-trip on Alaska Airlines for $275 a
  [5f86030e...] [ab8f1310...] role=agent  type=turn     Absolutely! NYC has incredible dining options

### 3d. Update & Delete in Cosmos DB

If the content changes, the embedding is automatically re-generated (awaited).

In [10]:
# Update the user's budget message to add a hotel budget constraint
user_msgs = await memory.get_memories(user_id="user-001", role="user")
target = [m for m in user_msgs if "Pike Place" in m["content"]][0]
print(f"Before: {target['content']}\n")

await memory.update_cosmos(
    memory_id=target["id"],
    content="Something near Pike Place Market would be great. Keep flights under $300 round trip and hotels under $200/night.",
    metadata={"edited": True, "reason": "user clarified hotel budget"},
)

updated = (await memory.get_memories(memory_id=target["id"]))[0]
print(f"After:  {updated['content']}")

Before: Something near Pike Place Market would be great. Keep flights under $300 round trip and hotels under $200/night.

After:  Something near Pike Place Market would be great. Keep flights under $300 round trip and hotels under $200/night.


In [13]:
# Delete the tool memory from Cosmos (async)
tool_mems = await memory.get_memories(user_id="user-002", role="tool")
print(tool_mems[0])
if tool_mems:
    await memory.delete_cosmos(
        tool_mems[0]["id"],
        thread_id=tool_mems[0]["thread_id"],
        user_id=tool_mems[0]["user_id"],
    )
    print(f"Deleted tool memory {tool_mems[0]['id'][:8]}...")

# Verify
remaining = await memory.get_memories()
print(f"\nRemaining memories in Cosmos DB: {len(remaining)}")
for r in remaining:
    print(f"  [{r['thread_id'][:8]}...] [{r['id'][:8]}...] role={r['role']:<6} type={r['type']:<8} {r['content'][:50]}")

get_memories returned empty results


IndexError: list index out of range

### 3e. Retrieve a full thread with `get_thread`

Same parameters: `thread_id` (required), `user_id` (optional), `recent_k` (optional).

In [14]:
# Pick a thread_id from an existing memory in Cosmos DB
sample = await memory.get_memories(user_id="user-001")
if sample:
    thread_id = sample[0]["thread_id"]
    print(f"Using thread_id: {thread_id}\n")

    # Get all documents in the thread
    thread_all = await memory.get_thread(thread_id=thread_id)
    print(f"All memories in thread: {len(thread_all)}")
    for m in thread_all:
        print(f"  [{m['thread_id'][:8]}...] [{m['id'][:8]}...] role={m['role']:<6} type={m['type']:<8} {m['content'][:60]}")

    # Get only the 2 most recent documents
    thread_recent = await memory.get_thread(thread_id=thread_id, recent_k=2)
    print(f"\nMost recent 2 memories:")
    for m in thread_recent:
        print(f"  [{m['thread_id'][:8]}...] [{m['id'][:8]}...] role={m['role']:<6} type={m['type']:<8} {m['content'][:60]}")

    # Filter by user_id as well
    thread_user = await memory.get_thread(thread_id=thread_id, user_id="user-001")
    print(f"\nThread memories for user-001: {len(thread_user)}")
else:
    print("No memories found in Cosmos DB – run the async add_cosmos cells first.")

Using thread_id: c55ab0cb-326a-478a-aaa2-af1469480f58

All memories in thread: 5
  [c55ab0cb...] [59ba9c05...] role=user   type=turn     What's the weather like in Seattle this weekend?
  [c55ab0cb...] [fed9f5ee...] role=agent  type=turn     This weekend Seattle will be around 55°F with partly cloudy 
  [c55ab0cb...] [98c43caa...] role=agent  type=turn     Sure! I found round-trip flights departing Friday evening an
  [c55ab0cb...] [6e10d230...] role=user   type=turn     Something near Pike Place Market would be great. Keep flight
  [c55ab0cb...] [b99ae6ee...] role=agent  type=turn     I found a round-trip on Alaska Airlines for $275 and two hot

Most recent 2 memories:
  [c55ab0cb...] [6e10d230...] role=user   type=turn     Something near Pike Place Market would be great. Keep flight
  [c55ab0cb...] [b99ae6ee...] role=agent  type=turn     I found a round-trip on Alaska Airlines for $275 and two hot

Thread memories for user-001: 5


## 4. Thread Summary

`AsyncCosmosMemoryClient.generate_thread_summary()` uses **aiohttp** to trigger the Azure Durable Function orchestrator and poll for completion without blocking the event loop.

> **Prerequisites** are the same as the sync demo (section 4) \u2014 the Functions host must be running locally (or deployed).

In [15]:
# Verify ADF endpoint is configured (set in the Setup cell above)
print(f"ADF endpoint: {memory.adf_endpoint}")
print(f"ADF key set:  {bool(memory.adf_key)}")

ADF endpoint: https://akatariaagentmemory.azurewebsites.net/api
ADF key set:  True


In [16]:

# Pick a thread_id + user_id from existing Cosmos data
sample_adf_async = await memory.get_memories(user_id="user-001")
if not sample_adf_async:
    raise RuntimeError("No memories for user-001 in Cosmos DB. Run the add_cosmos cells first.")

thread_id_adf = sample_adf_async[0]["thread_id"]
user_id_adf = sample_adf_async[0]["user_id"]
print(f"Summarizing thread_id={thread_id_adf}  user_id={user_id_adf}\n")

# Call generate_thread_summary (all memories in the thread)
result_async = await memory.generate_thread_summary(user_id=user_id_adf, thread_id=thread_id_adf)

Summarizing thread_id=c55ab0cb-326a-478a-aaa2-af1469480f58  user_id=user-001



In [17]:
print("Status:", result_async.get("runtimeStatus"))
print("\nSummary document:")
print(json.dumps(result_async.get("output", {}), indent=2))

Status: Completed

Summary document:
{
  "id": "summary_user-001_c55ab0cb-326a-478a-aaa2-af1469480f58",
  "user_id": "user-001",
  "thread_id": "c55ab0cb-326a-478a-aaa2-af1469480f58",
  "role": "system",
  "type": "summary",
  "content": "**Summary:** The user inquired about the weather in Seattle for the upcoming weekend and then shifted to planning a trip, requesting affordable flights and accommodations near Pike Place Market. The assistant provided weather information and identified flight and hotel options within the user\u2019s specified budget. No reservations were finalized.\n\n**Key Points:**\n- The assistant reported weekend weather in Seattle: approximately 55\u00b0F, partly cloudy on Saturday, and light rain on Sunday.\n- The user expressed interest in traveling and asked about flights and hotels.\n- The user specified constraints: flights under $300 round trip and hotels under $200 per night, preferably near Pike Place Market.\n- The assistant found a round-trip Alaska Air

## 5. Fact Extraction
`extract_facts` triggers the same orchestrator but calls the **extract_facts** activity instead of generate_thread_summary. The LLM extracts discrete factual statements from the thread and stores them in Cosmos DB as a memory with `type: "fact"`.

> **Prerequisites** are the same as section 4 — the Functions host must be running.

In [18]:
# Extract facts from the same thread (async)
facts_result_async = await memory.extract_facts(user_id=user_id_adf, thread_id=thread_id_adf)

In [19]:
output = facts_result_async.get("output", {})
print(f"\nExtracted {len(output)} fact(s):\n")
for fact in output:
    print(f"  [{fact.get('thread_id', '')[:8]}...] user={fact.get('user_id', ''):<10} type={fact.get('type', ''):<8} {fact.get('content', '')[:]}")


Extracted 5 fact(s):

  [c55ab0cb...] user=user-001   type=fact     The assistant reported that Seattle’s weekend weather will be around 55°F, partly cloudy on Saturday, and light rain on Sunday.
  [c55ab0cb...] user=user-001   type=fact     The user requested round-trip flights under $300 and hotels under $200 per night near Pike Place Market.
  [c55ab0cb...] user=user-001   type=fact     The assistant found a round-trip Alaska Airlines flight for $275.
  [c55ab0cb...] user=user-001   type=fact     The assistant identified two hotels within a 5-minute walk of Pike Place Market: Inn at the Market ($189/night) and Hilton Garden Inn ($145/night).
  [c55ab0cb...] user=user-001   type=fact     No flight or hotel reservations were finalized in the conversation.


## 6. User Summary

`generate_user_summary` triggers the orchestrator to aggregate memories **across all threads** for a user and produce a structured profile covering preferences, account state, compliance details, and behavioural patterns. The result is stored in Cosmos DB as a memory with `type: "user_summary"`.

Retrieve the stored user summary at any time with `get_user_summary(user_id)` — useful for priming new conversations with user context.

> **Prerequisites** are the same as sections 4 and 5 — the Functions host must be running.

In [20]:
# Generate a user summary across all threads (async)
user_summary_result = await memory.generate_user_summary(user_id=user_id_adf)

print("Status:", user_summary_result.get("runtimeStatus"))
print("\nUser Summary document:")
print(json.dumps(user_summary_result.get("output", {}), indent=2))

Status: Completed

User Summary document:
{
  "id": "user_summary_user-001",
  "user_id": "user-001",
  "thread_id": "__user_summary__",
  "role": "system",
  "type": "user_summary",
  "content": "## Personal Preferences\n- The user specifies clear budget constraints when planning travel (e.g., flights under $300 round trip and hotels under $200 per night).\n- The user prefers accommodations in specific, walkable locations (e.g., within a 5-minute walk of Pike Place Market in Seattle).\n- The user is open to specific airline and hotel suggestions as long as they meet stated price and location criteria.\n\n---\n\n## Goals & Current Work\n- The user is planning a weekend trip to Seattle.\n- The user is seeking round-trip flights under $300.\n- The user is seeking hotel accommodations under $200 per night near Pike Place Market.\n- As of the latest thread, no flight or hotel reservations have been finalized.\n\n---\n\n## Behavioral Patterns\n- The user refines requests after initial sugge

In [21]:
# Retrieve the stored user summary from Cosmos DB (async)
stored_summary = await memory.get_user_summary(user_id=user_id_adf)
if stored_summary:
    print("User Summary for", user_id_adf)
    print(stored_summary[0]["content"])
else:
    print("No user summary found — run the generate_user_summary cell first.")

User Summary for user-001
## Personal Preferences
- The user specifies clear budget constraints when planning travel (e.g., flights under $300 round trip and hotels under $200 per night).
- The user prefers accommodations in specific, walkable locations (e.g., within a 5-minute walk of Pike Place Market in Seattle).
- The user is open to specific airline and hotel suggestions as long as they meet stated price and location criteria.

---

## Goals & Current Work
- The user is planning a weekend trip to Seattle.
- The user is seeking round-trip flights under $300.
- The user is seeking hotel accommodations under $200 per night near Pike Place Market.
- As of the latest thread, no flight or hotel reservations have been finalized.

---

## Behavioral Patterns
- The user refines requests after initial suggestions (e.g., clarifying hotel budget and preferred neighborhood).
- The user responds concisely and focuses on constraints (price and location) rather than amenities or brand preferences

### 7. Vector search with `search_cosmos`

Same as the sync version — embeds the query and runs a `VectorDistance` similarity search.

In [22]:
results_search_async = await memory.search_cosmos(
    search_terms="What did the user ask about the weather?",
    user_id="user-001",
    top_k=3, hybrid_search= True
)

In [23]:
print(f"Top {len(results_search_async)} results:\n")
for r in results_search_async:
    score = r.get("score", "N/A")
    print(f"  [{r['id'][:8]}...] score={score}  {r['content'][:60]}")

Top 3 results:

  [summary_...] score=N/A  **Summary:** The user inquired about the weather in Seattle 
  [ee0510a6...] score=N/A  The assistant reported that Seattle’s weekend weather will b
  [user_sum...] score=N/A  ## Personal Preferences
- The user specifies clear budget co


### 8. Automatic Processing (Change Feed)

The toolkit includes a Cosmos DB change feed trigger that automatically fires thread summaries, fact extraction, and user summaries when configurable message count thresholds are crossed.

**Prerequisites:**
- The Azure Functions host must be running (`func start`)
- `local.settings.json` must include change feed settings:
  - `COSMOS_DB_CONNECTION__accountEndpoint` pointing to your Cosmos account
  - `COSMOS_DB_COUNTERS_CONTAINER` set to `"counters"`
  - At least one threshold > 0 (e.g. `THREAD_SUMMARY_EVERY_N=3`)
- A `counters` container must exist in the same database (partition key: `/user_id`)

The cells below write enough turns to cross the threshold, then poll for auto-generated derived memories.

In [ ]:
# Write turns to trigger automatic processing
# Assumes THREAD_SUMMARY_EVERY_N=3 in local.settings.json
import uuid, asyncio

changefeed_thread = str(uuid.uuid4())
print(f"Thread ID: {changefeed_thread}")

for i in range(3):
    await memory.add_cosmos(
        user_id="user-001",
        thread_id=changefeed_thread,
        role="user",
        content=f"Change feed test message {i+1}: Tell me about vector databases.",
    )
    print(f"  Wrote turn {i+1}")

print("\nTurns written. The change feed trigger should fire within a few seconds.")
print("Waiting 15 seconds for processing...")
await asyncio.sleep(15)

In [ ]:
# Check for auto-generated derived memories
results = await memory.get_memories(user_id="user-001", thread_id=changefeed_thread, memory_type="summary")
if results:
    print("Auto-generated thread summary found!")
    for r in results:
        print(f"  [{r['id'][:20]}...] {r['content'][:100]}")
else:
    print("No summary yet — the change feed may need more time. Try re-running this cell after a few seconds.")

print()

facts = await memory.get_memories(user_id="user-001", thread_id=changefeed_thread, memory_type="fact")
if facts:
    print(f"Auto-extracted facts ({len(facts)}):")
    for f in facts:
        print(f"  • {f['content'][:80]}")
else:
    print("No facts yet — check FACT_EXTRACTION_EVERY_N in local.settings.json.")

In [24]:
# Clean up async clients when done
await memory.close()
print("Async clients closed.")

Async clients closed.
